# 3단계 · 아이템 상세 수집

**목표:** 연관 아이템 각각의 **지표(기술집약도·수요/공급 부상도)**를 수집합니다.

## 핵심: `network/list` 한 번의 호출로 끝
- `/items/{id}/network/list?degree=N` (A4.5) 는 시드의 연관 아이템 **전체를 지표까지 붙여** 한 번에 돌려줍니다.
- 각 항목: `{rank, itemName, category, techIntensity, demandEmergence, supplyEmergence}`

## 왜 상세조회(A4.3) 대신 network/list?
| | 상세조회 A4.3 (아이템마다) | **network/list A4.5 (1콜)** |
|---|---|---|
| 호출 수 | 2차수 시 ~97번 | **1번** |
| 지표 | 수요·공급 2종 | **기술집약도까지 3종** |
| 속도 | 느림(~1~2분) | 빠름(~1초) |

→ 교육툴은 **network/list 1콜**로 대체해 빠르고 지표도 풍부합니다.

> **1단계에서 확정한 시드를 사용합니다.**
> 데모 시드: **Mathematical finance** (itemId `33589680`, 카테고리 *인공지능*, 지표 *기술집약도*)

In [ ]:
# --- APOLLO API 호출 헬퍼 (모든 노트북 공통) ---
import requests, urllib3
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://apollo.kisti.re.kr/service-test"   # 테스트 서버

def call_api(method, path, params=None, payload=None):
    """APOLLO 호출 후 {meta, data} 본문을 반환. (테스트 서버라 verify=False)"""
    r = requests.request(method, BASE_URL + path, params=params, json=payload,
                         headers={"Content-Type": "application/json"},
                         verify=False, timeout=120)
    body = r.json()
    if not (isinstance(body, dict) and "data" in body):
        raise RuntimeError(f"API 오류 (HTTP {r.status_code}): {body}")
    return body

print("준비 완료 · BASE_URL =", BASE_URL)

In [ ]:
# 1단계에서 확정한 시드 (다른 데모 사례로 바꾸려면 값만 교체)
SEED_ID = 33589680          # Mathematical finance
SEED_NAME = "Mathematical finance"
CATEGORY = "인공지능"
DEGREE = 3                  # APOLLO degree=3  →  사용자 체감 '2차수'(이웃의 이웃)

### network/list 호출 (A4.5)

In [ ]:
res = call_api("GET", f"/open/api/v1/items/{SEED_ID}/network/list",
               params={"degree": DEGREE})
lst = res["data"]

df = pd.DataFrame(lst)
df.columns = ["순위", "아이템명", "카테고리", "기술집약도", "수요부상성", "공급부상성"]
print(f"연관 아이템 {len(df)}건의 지표를 1회 호출로 수집")
df.head(15)

### 지표 분포 훑어보기

In [ ]:
df[["기술집약도", "수요부상성", "공급부상성"]].describe().round(2)

**정리**
- 한 번의 호출로 연관 아이템 전체의 3지표를 확보했습니다.
- 4단계에서 이 네트워크를 **시각화**하고, 5단계에서 이 지표로 **종합 스코어링**합니다.